# Fine-Tune SAM2.1 for Segmentation

SAM2 is a state-of-the-art segmentation model developed by Meta. Link: We can fine-tune a pre-trained model on our custom data.

Using the trained model, we can then run inference on our test dataset of real tactile paving data to get a picture of how the model's segmentations would work in real life scenarios.

This notebook is based off the forground segmentation notebook provided by Roboflow here: [link](https://blog.roboflow.com/fine-tune-sam-2-1/)

## Installation

Clone the repository

In [ ]:
!git clone https://github.com/facebookresearch/sam2.git

Download the optimized YAML file coniguration for training

In [ ]:
!wget -O /content/sam2/sam2/configs/train.yaml 'https://drive.usercontent.google.com/download?id=11cmbxPPsYqFyWq87tmLgBAQ6OZgEhPG3'

Install SAM2

In [ ]:
%cd ./sam2/

In [ ]:
!pip install -e .[dev] -q

Download the pre-trained checkpoints

In [ ]:
!cd ./checkpoints && ./download_ckpts.sh

## Data

Please request permission access from the forms provided to download both our custom synthetically augmented dataset as well as our collected real-world testset that will be used for training and evalutation respectively.

In order to train the model, the datasets must be arranged in a specific format. For SAM2.1, the preferred data structure format required for training is of the form:
```
train
    - image1.jpg
    - image1.json ...
val
    - image100.jpg
    - image100.json ...
```
Where the json files contain the corresponding image filenames and segmentation run-length-encodings.

**IMPORTANT NOTE BEFORE TRAINING:** 
- If errors persist in installation, update in pyproject.toml:  
"setuptools>=61.0",
to
"setuptools>=62.3.0,<75.9",

- Update in train.yaml file: 

Dataset Location Path

Ground Truth Location Path

Checkpoint file that is preffered to train (small, base-plus, etc.)

OPTIONAL: Batch-size, Train workers, Epochs etc. 

## Training

You can now start training a SAM-2.1 model. The amount of time it will take to train the model will vary depending on the GPU you are using and the number of images in your dataset.

In [ ]:
!python training/train.py -c 'configs/train.yaml' --use-cluster 0 --num-gpus 1

## INFERENCE

With a trained model ready, we can test the model on an image from our test set.

To assist with visualizing model predictions, we are going to use Roboflow supervision, an open source computer vision Python package with utilities for working with vision model outputs

In [ ]:
!pip install supervision -q

### Load model from checkpoint

In [ ]:
import torch
from sam2.build_sam import build_sam2
from sam2.automatic_mask_generator import SAM2AutomaticMaskGenerator
import supervision as sv
import os
import random
from PIL import Image
import numpy as np

# use bfloat16 for the entire notebook
# from Meta notebook
torch.autocast("cuda", dtype=torch.bfloat16).__enter__()
if torch.cuda.get_device_properties(0).major >= 8:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

checkpoint = ".../checkpoint.pt" # IMP: CHANGE TO PREFERRED CHECKPOINT! 
model_cfg = "configs/sam2.1/sam2.1_hiera_b+.yaml"
sam2 = build_sam2(model_cfg, checkpoint, device="cuda")
mask_generator = SAM2AutomaticMaskGenerator(sam2)

checkpoint_base = "/content/sam2/checkpoints/sam2.1_hiera_base_plus.pt"
model_cfg_base = "configs/sam2.1/sam2.1_hiera_b+.yaml"
sam2_base = build_sam2(model_cfg_base, checkpoint_base, device="cuda")
mask_generator_base = SAM2AutomaticMaskGenerator(sam2_base)

### Run Inference on an Image in Automatic Mask Generation Mode

In [ ]:
validation_set = os.listdir("/content/data/valid")

# choose random with .json extension
image = random.choice([img for img in validation_set if img.endswith(".jpg")])
image = os.path.join("/content/data/valid", image)
opened_image = np.array(Image.open(image).convert("RGB"))
result = mask_generator.generate(opened_image)

detections = sv.Detections.from_sam(sam_result=result)

mask_annotator = sv.MaskAnnotator(color_lookup = sv.ColorLookup.INDEX)
annotated_image = opened_image.copy()
annotated_image = mask_annotator.annotate(annotated_image, detections=detections)

base_annotator = sv.MaskAnnotator(color_lookup = sv.ColorLookup.INDEX)

base_result = mask_generator_base.generate(opened_image)
base_detections = sv.Detections.from_sam(sam_result=base_result)
base_annotated_image = opened_image.copy()
base_annotated_image = base_annotator.annotate(base_annotated_image, detections=base_detections)

sv.plot_images_grid(images=[annotated_image, base_annotated_image], grid_size=(1, 2))